[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03a_baseline.ipynb)

# 03a — Baseline Experiment: Random Search and a Rule-Based Sweep

## 1. Overview

This notebook evaluates two **baseline optimization methods** for multi-band antenna tilt coordination: choosing one
tilt per `(cell, band)` pair to improve coverage while limiting co-band overlap. The baselines are
the reference TuRBO (notebook 03b) is read against.

**Goals.**

- Run two simple, reproducible baselines on the same simulator, scenario and objective as TuRBO.
- Measure what they change on every KPI, and where on the map.
- Set the reference performance level for the method comparison in notebook 04.

The problem formulation (the tilt vector and its bounds) and the KPI definitions are described in notebook 01
(sections 2, 7, 8 and 12) and are not repeated here.

**Outputs.** One run directory per method and seed under `outputs/optim/`, tables under
`reports/tables/03a_baseline/`, figures under `reports/figures/03a_baseline/`. **Requirements.** A CUDA GPU.
The shell equivalent of the runs is `task baseline` (random search) and `task baseline -- optim/method=rule`.

In [1]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna.rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [2]:
%load_ext autoreload
%autoreload 2

from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.config import load_config
from src.evaluation import compare, maps, plots
from src.evaluation import runs as run_store
from src.evaluation.export import readable, save_table
from src.kpi.capacity import CapacitySpec, max_rsrp
from src.kpi.overlap import overlap_neighbors
from src.optim.objective import MAXIMISED
from src.optim.run import run
from src.optim.space import TiltSpace
from src.utils.plotting import label, save_fig, setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/03a_baseline"))
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path("reports/tables/03a_baseline"))
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

# Search seeds: optim.seed from the config; BAND_TILT_SEEDS (space-separated) overrides it.
SEEDS = [int(seed) for seed in os.environ.get("BAND_TILT_SEEDS", str(cfg.optim.seed)).split()]
METHODS = ("random", "rule")

## 2. Baseline Methods

### Random search (`optim/method=random`)

Scrambled Sobol points drawn uniformly over the tilt box, each ray-traced and scored; the configuration with the
highest objective is kept. It is matched to TuRBO on the evaluation budget and seed, and its first `n_init` points
are TuRBO's initial design, so any gap between the two is what the model contributes.

### Rule-based sweep (`optim/method=rule`)

The operator heuristic: every cell on a band shares one tilt, and the band tilts are tuned by coordinate descent,
trying `n_steps` values per band per pass over `n_rounds` passes. It is deterministic and not budget-matched.

### Why these baselines?

- Neither needs a learned model.
- Both are straightforward to reproduce from a seed and a config.
- Random search makes no assumption about the objective landscape; the sweep encodes the one an operator would make.
- Together they bound what TuRBO has to beat: an unstructured search at equal budget and a structured one at far
  lower cost.

## 3. Experimental Setup

Every value below is read from the composed config and the simulation output, not restated by hand.

In [3]:
space = TiltSpace.from_config(cfg)
baseline = run_store.baseline_map(cfg)
cells = pd.read_parquet(cfg.data.output.cell_file).drop_duplicates("cell")
ue = pd.read_parquet(cfg.data.output.ue_file)
radio_cfg = cfg.simulation.radio_map
n_rows, n_cols = maps.grid_shape(baseline)

network = pd.DataFrame(
    {
        "Parameter": [
            "Sites (nodes)",
            "Cells",
            "Frequency bands",
            "Optimized variables",
            "Tilt range [°]",
            "Current tilt [°]",
            "Tilt step, maximum tilt change",
        ],
        "Value": [
            cells["node"].nunique(),
            len(space.cells),
            ", ".join(label(band) for band in space.band_names),
            f"{space.n_dim} ({len(space.cells)} cells x {len(space.band_names)} bands)",
            f"{space.lower.min():g} – {space.upper.max():g}",
            ", ".join(f"{value:g}" for value in np.unique(space.baseline)),
            "Not constrained: continuous tilt inside the box",
        ],
    }
)
mechanisms = [
    name
    for name in ("los", "specular_reflection", "diffuse_reflection", "refraction", "diffraction")
    if bool(radio_cfg[name])
]
simulation = pd.DataFrame(
    {
        "Parameter": [
            "Simulator",
            "Scene",
            "Scenario",
            "Grid resolution [m]",
            "Evaluation tiles",
            "Rays per transmitter",
            "Maximum path depth",
            "Propagation mechanisms",
        ],
        "Value": [
            "Sionna RT",
            str(cfg.simulation.scene.name),
            str(baseline["scenario_id"]),
            f"{float(baseline['tile_size_m']):g}",
            f"{n_rows * n_cols} ({n_rows} x {n_cols})",
            f"{int(radio_cfg.samples_per_tx):,}",
            int(radio_cfg.max_depth),
            ", ".join(mechanisms),
        ],
    }
)
users = pd.DataFrame(
    {
        "Parameter": ["UE reports", "Intervals", "Distribution"],
        "Value": [
            len(ue),
            ue["t_index"].nunique(),
            "Uniform open-ground background plus Gaussian hotspots (notebook 00)",
        ],
    }
)
for name, table in (("network", network), ("simulation", simulation), ("users", users)):
    save_table(table, f"setup_{name}")
    display(table)

,Parameter,Value
0,Sites (nodes),4
1,Cells,12
2,Frequency bands,"2600 MHz, 1800 MHz, 700 MHz"
3,Optimized variables,36 (12 cells x 3 bands)
4,Tilt range [°],0 – 15
5,Current tilt [°],"8, 10, 12"
6,"Tilt step, maximum tilt change",Not constrained: continuous tilt inside the box


,Parameter,Value
0,Simulator,Sionna RT
1,Scene,data/external/scene/scene.xml
2,Scenario,scn_7d938e15f9ac4618
3,Grid resolution [m],20
4,Evaluation tiles,101060 (326 x 310)
5,Rays per transmitter,"10,000,000"
6,Maximum path depth,8
7,Propagation mechanisms,"los, specular_reflection, refraction"


,Parameter,Value
0,UE reports,10087
1,Intervals,672
2,Distribution,Uniform open-ground background plus Gaussian h...


Every candidate is scored on all UEs, so the numbers from section 6 onwards are the search's own measurements.

## 4. Baseline Configuration

Both methods start from the committed network configuration, which is always evaluation 0 of a run.

In [4]:
def method_config(method):
    return load_config(overrides=[f"optim/method={method}", *CONFIG_OVERRIDES]).optim


random_cfg, rule_cfg = method_config("random"), method_config("rule")
n_band = len(space.band_names)
budgets = pd.DataFrame(
    {
        "Parameter": [
            "Search strategy",
            "Evaluations (excluding the incumbent)",
            "Budget split",
            "Search seeds",
            "Initial configuration",
            "Solutions published",
        ],
        label("random"): [
            "Scrambled Sobol sampling over the tilt box",
            int(random_cfg.method.budget.n_init) + int(random_cfg.method.budget.n_iter),
            f"n_init={random_cfg.method.budget.n_init}, n_iter={random_cfg.method.budget.n_iter}",
            ", ".join(map(str, SEEDS)),
            "Current network",
            int(random_cfg.n_solutions),
        ],
        label("rule"): [
            "One tilt per band, coordinate descent",
            f"at most {n_band * int(rule_cfg.method.n_steps) * int(rule_cfg.method.n_rounds)}",
            f"n_steps={rule_cfg.method.n_steps}, n_rounds={rule_cfg.method.n_rounds}",
            "Deterministic",
            "Current network",
            int(rule_cfg.n_solutions),
        ],
    }
)
save_table(budgets, "baseline_configuration")
budgets

,Parameter,Random search,Rule-based sweep
0,Search strategy,Scrambled Sobol sampling over the tilt box,"One tilt per band, coordinate descent"
1,Evaluations (excluding the incumbent),144,at most 30
2,Budget split,"n_init=16, n_iter=128","n_steps=5, n_rounds=2"
3,Search seeds,42,Deterministic
4,Initial configuration,Current network,Current network
5,Solutions published,4,4


## 5. Runs

Each run evaluates the current configuration first, then searches, then re-traces its winner once to archive its
radio map. A run already on disk for the same method and seed is reused; delete its directory under `outputs/optim/` to
repeat it. The runs are then checked comparable with the current radio map: same scenario, grid, solver settings,
bands and KPI definition.

In [5]:
def on_disk():
    # The newest finished run of each method and seed.
    return run_store.latest_per_method_and_seed(run_store.discover(cfg.optim.output.dir))


done = {(r.method, r.seed) for r in on_disk()}
for seed in SEEDS:
    if ("random", seed) not in done:
        run(load_config(overrides=["optim/method=random", f"optim.seed={seed}", *CONFIG_OVERRIDES]))

# Deterministic, so one run; its seed only labels it.
if not any(method == "rule" for method, _ in done):
    run(load_config(overrides=["optim/method=rule", f"optim.seed={SEEDS[0]}", *CONFIG_OVERRIDES]))

runs = [r for r in on_disk() if r.method == "rule" or (r.method == "random" and r.seed in SEEDS)]
checks = run_store.verify(runs, baseline)
run_store.require(checks)
print(f"{len(runs)} runs; {int(checks['holds'].sum())} of {len(checks)} comparability checks hold")
# Each method's best run over its seeds, for the per-method sections below.
best = compare.best_run_per_method(runs)

TypeError: verify() missing 1 required positional argument: 'cfg'

## 6. Initial Network State

The committed configuration, scored on every UE. Mean overlapping neighbours (co-band cells within the overlap margin
of the serving cell, summed over bands, averaged over covered tiles) is a diagnostic computed from the radio map; no
score reads it.

In [ ]:
def mean_overlap_neighbors(rsrp):
    covered = max_rsrp(rsrp) > float(cfg.kpi.hole_dbm)
    return float(overlap_neighbors(rsrp, cfg)[covered].mean())


incumbent = runs[0].incumbent_kpi
baseline_rsrp = baseline["rsrp_dbm"].astype(float)
initial = pd.DataFrame(
    {
        "kpi": [*incumbent.as_dict(), "Mean overlapping neighbours"],
        "direction": [compare.direction(name) for name in incumbent.as_dict()] + ["minimise"],
        "value": [
            *incumbent.as_dict().values(),
            mean_overlap_neighbors(baseline_rsrp),
        ],
    }
)
initial = readable(initial.rename(columns={"value": "Initial value"}))
save_table(initial, "initial_state")
initial

,KPI,Direction,Initial value
0,Coverage hole rate,minimise,0.1125
1,Co-band overlap rate,minimise,0.3164
2,Overlap neighbours per covered tile,minimise,0.9625
3,Weak coverage rate,minimise,0.3069
4,"Cell-edge RSRP, p05 [dBm]",maximise,-108.5644
5,"Median RSRP, p50 [dBm]",maximise,-84.1068
6,"Cell-edge SINR, p05 [dB]",maximise,-5.8729
7,"Median SINR, p50 [dB]",maximise,8.4932
8,Served UE rate,maximise,0.5265
9,Peak PRB utilisation,minimise,0.7999


## 7. Optimization Process and Objective

Every candidate goes through the same loop, and every KPI is a measurement: nothing is predicted by a surrogate.

```text
Current tilt configuration (evaluation 0)
        │
        ▼
Generate candidate tilt ── random: Sobol point · rule: one band's shared tilt
        │
        ▼
Apply constraints ──────── clip to [tilt_min, tilt_max] per cell-band
        │
        ▼
Radio simulation ───────── Sionna RT, one solve per band
        │
        ▼
Compute KPIs and J ─────── src/optim/objective.py
        │
        ▼
Keep best configuration ── highest J; a tie keeps the earlier evaluation
```

Both methods maximise the project objective ([ADR 0009](../docs/adr/0009-effective-coverage-objective.md)):

$$
J = \frac{\sum_{g \in G} w_g\, \lambda_g\, e^{1 - \lambda_g}}{\sum_{g \in G} w_g},
\qquad
\lambda_g = 1 + m_{b(g)}(g),
\qquad
w_g = 1 + r_g
$$

$b(g)$ is the most preferred band whose strongest cell clears $T_\text{cov}$ —
the `kpi.capacity.band_preference` order the serving rule admits UEs by — and
$m_{b(g)}$ counts the other cells **on that band** above $T_\text{cov}$ and
within $\Delta_R$ of its strongest. So $\lambda_g$ is the number of cells
contending to serve tile $g$, and $\lambda_g = 0$ where no band covers it.

$\lambda e^{1 - \lambda}$ is worth exactly **1 at $\lambda = 1$**, 0.736 at 2,
0.406 at 3 and 0 at 0: the objective pays for one dominant server and for nothing
else. $r_g$ is the tile's MDT reports over the busiest tile's, so
$w_g \in [1, 2]$ — demand doubles a tile at most and silences none.

$J$ is therefore a weighted average of values in $[0, 1]$, so **$J \in [0, 1]$**
and **higher is better**; 1 would mean every tile of the grid has exactly one
serving cell. The objective has no free parameters. The KPIs are reported beside
it, tile-uniform and band-collapsed; none of them is weighted into it.

In [ ]:
# The objective has no parameters of its own (ADR 0009): it reads the KPI
# thresholds and the serving band preference, and nothing else.
parameters = pd.DataFrame(
    {
        "Parameter": ["T_cov [dBm]", "Delta_R [dB]", "Band priority", "J range"],
        "Value": [
            cfg.kpi.hole_dbm,
            cfg.kpi.overlap_margin_db,
            " > ".join(str(band) for band in cfg.kpi.capacity.band_preference),
            "[0, 1], maximised",
        ],
    }
)
save_table(parameters, "objective_parameters")
parameters

,Parameter,Value
0,T_cov [dBm],-120.0
1,Delta_R [dB],6.0
2,Band priority,b2600 > b1800 > b700
3,J range,"[0, 1], maximised"


## 8. Optimization Progress

Best value found so far per evaluation, from the search history. Random search is shown as the mean over seeds with
the min–max band.

In [ ]:
trace = compare.convergence(runs)
figure = plots.convergence_plot(trace)
save_fig(figure, "search_progress")
plt.show()

figure, axes = plt.subplots(2, 2, figsize=(12.0, 7.0), constrained_layout=True)
for axis, name in zip(axes.ravel(), ("hole_rate", "overlap_rate", "weak_rate", "served_rate"), strict=True):
    part = trace[trace["kpi"] == name]
    for method, group in part.groupby("method", sort=False):
        mean = group.groupby("iteration")["value"].mean()
        axis.plot(mean.index, mean.to_numpy(), color=plots.COLOURS.get(method), label=label(method))
    axis.set_title(f"{label(name)} ({compare.direction(name)})")
    axis.set_xlabel("Evaluations")
    axis.set_ylabel("Best so far")
axes[0, 0].legend()
figure.suptitle("Best KPI value so far, each KPI tracked on its own")
save_fig(figure, "kpi_progress")
plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\1173306165.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\1173306165.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Each KPI panel tracks that KPI's own best value, which need not come from the configuration with the best J.

**Observations.** Random search found its best configuration at evaluation 62 and none of the later 83 draws beat
it; its best J is 0.8155 against the current configuration's 0.8080 (+0.92 %). The rule-based sweep supplies its
whole gain in a first pass over the three bands - 0.8189 by evaluation 11, and the remaining 17 evaluations do not
improve on it. It ends **above** random search on J at a seventh of the ray-tracing cost (1.1 minutes against 7.5),
and it is ahead of TuRBO too at any budget up to 50 evaluations (notebook 04).

$J$ is the demand-weighted share of the grid served cleanly by exactly one cell (ADR 0009), so it is bounded in
$[0, 1]$ and these values compare with nothing recorded before that change. The headroom is therefore small by
construction: from 0.8080 the most any configuration could gain is 0.1920, and the sweep takes 5.7 % of that.
Read the gain beside its cost: **overlap neighbours per covered tile worsen under both** (+16.4 % for the sweep and
+9.1 % for random search), because the objective pays 1.000 for a tile lifted out of a hole and charges only 0.264
for a clean tile split in two - one closed hole is worth 3.78 newly crowded ones.

## 9. Best Tilt Configuration

The deliverable of each method's best run: current and proposed tilt per cell-band. Negative change is uptilt.

In [ ]:
for method in METHODS:
    chosen = best[method]
    tilt = readable(chosen.best_tilt)
    save_table(tilt, f"best_tilt_{method}")
    summary = readable(compare.tilt_movement(chosen))
    save_table(summary, f"tilt_movement_{method}")
    print(f"{label(method)}: run {chosen.run_id}, seed {chosen.seed}, best evaluation {chosen.best_index}")
    display(summary)
    display(tilt)
    figure = plots.tilt_movement_plot(chosen.best_tilt, method)
    save_fig(figure, f"tilt_movement_{method}")
    plt.show()

Random search: run 2026-09-20_04-09-52, seed 42, best evaluation 62


,Band,Cells,Cells moved,Mean absolute tilt change [°],Largest tilt change [°],Mean tilt change [°]
0,1800 MHz,12,12,3.8748,8.5457,-3.8177
1,2600 MHz,12,12,6.1655,11.6103,-5.2764
2,700 MHz,12,12,3.6253,7.4301,-1.9856


,Cell,Band,Current tilt [°],Proposed tilt [°],Tilt change [°],Minimum tilt [°],Maximum tilt [°]
0,n0c0,2600 MHz,12.0,0.3897,-11.6103,0.0,15.0
1,n0c0,1800 MHz,10.0,8.7017,-1.2983,0.0,15.0
2,n0c0,700 MHz,8.0,11.2512,3.2512,0.0,15.0
3,n0c1,2600 MHz,12.0,4.7738,-7.2262,0.0,15.0
4,n0c1,1800 MHz,10.0,9.2875,-0.7125,0.0,15.0
5,n0c1,700 MHz,8.0,10.9761,2.9761,0.0,15.0
6,n0c2,2600 MHz,12.0,14.7036,2.7036,0.0,15.0
7,n0c2,1800 MHz,10.0,4.5199,-5.4801,0.0,15.0
8,n0c2,700 MHz,8.0,2.0646,-5.9354,0.0,15.0
9,n1c0,2600 MHz,12.0,14.6312,2.6312,0.0,15.0


Rule-based sweep: run 2026-09-20_04-20-14, seed 42, best evaluation 11


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2859297502.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Band,Cells,Cells moved,Mean absolute tilt change [°],Largest tilt change [°],Mean tilt change [°]
0,1800 MHz,12,12,10.0,10.0,-10.0
1,2600 MHz,12,12,4.5,4.5,-4.5
2,700 MHz,12,12,8.0,8.0,-8.0


,Cell,Band,Current tilt [°],Proposed tilt [°],Tilt change [°],Minimum tilt [°],Maximum tilt [°]
0,n0c0,2600 MHz,12.0,7.5,-4.5,0.0,15.0
1,n0c0,1800 MHz,10.0,0.0,-10.0,0.0,15.0
2,n0c0,700 MHz,8.0,0.0,-8.0,0.0,15.0
3,n0c1,2600 MHz,12.0,7.5,-4.5,0.0,15.0
4,n0c1,1800 MHz,10.0,0.0,-10.0,0.0,15.0
5,n0c1,700 MHz,8.0,0.0,-8.0,0.0,15.0
6,n0c2,2600 MHz,12.0,7.5,-4.5,0.0,15.0
7,n0c2,1800 MHz,10.0,0.0,-10.0,0.0,15.0
8,n0c2,700 MHz,8.0,0.0,-8.0,0.0,15.0
9,n1c0,2600 MHz,12.0,7.5,-4.5,0.0,15.0


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2859297502.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Observations.** The sweep sets every cell to 7.5° on 2600 MHz and **0° on both 1800 and 700 MHz**: 4.5° of
uptilt on the high band, 10° on the mid and 8° on the low, applied identically to all twelve cells, since it can only
move bands as a whole. Both lower bands end pinned at the bottom of their tilt box, so the sweep would take them
further if the bounds allowed - the [0°, 15°] box is binding on two of its three dimensions.

Random search's best point also uptilts every band on average - to 6.7° on 2600 MHz, 6.2° on 1800 MHz and 6.0° on
700 MHz - but by amounts that vary from 0.4° to 14.7° with no visible pattern by node. That spread is what one Sobol
draw looks like, not evidence that those cells need different tilts.

## 10. Final Results

### KPI comparison

Each method's best configuration against the current one, on every UE. Relative improvement is signed so that
positive is better whichever direction the KPI runs:

$$
\text{Improvement} = s \cdot \frac{KPI_\text{optimized} - KPI_\text{initial}}{|KPI_\text{initial}|} \times 100\%,
\qquad s = \begin{cases} +1 & \text{maximised} \\ -1 & \text{minimised} \end{cases}
$$

In [ ]:
def kpi_comparison(chosen):
    table = compare.delta_table(chosen.incumbent_kpi, chosen.best_kpi)
    sign = np.where(table["kpi"].isin(MAXIMISED), 1.0, -1.0)
    table["improvement_pct"] = 100.0 * sign * table["delta"] / table["before"].abs()
    return table


for method in METHODS:
    table = readable(kpi_comparison(best[method])).rename(
        columns={
            "direction": "Direction",
            "before": "Initial",
            "after": label(method),
            "delta": "Change",
            "verdict": "Verdict",
            "improvement_pct": "Improvement [%]",
        }
    )
    save_table(table, f"kpi_comparison_{method}")
    print(label(method))
    display(table)

cost = readable(compare.method_table(runs).drop(columns=["run"]))
save_table(cost, "baseline_results")
cost

Random search


,KPI,Direction,Initial,Random search,Change,Verdict,Improvement [%]
0,Coverage hole rate,minimise,0.1125,0.1094,-3.0873e-03,better,2.7445
1,Co-band overlap rate,minimise,0.3164,0.3368,2.0414e-02,worse,-6.4519
2,Overlap neighbours per covered tile,minimise,0.9625,1.0501,8.7570e-02,worse,-9.0979
3,Weak coverage rate,minimise,0.3069,0.3063,-6.6297e-04,better,0.2160
4,"Cell-edge RSRP, p05 [dBm]",maximise,-108.5644,-108.4613,1.0307e-01,better,0.0949
5,"Median RSRP, p50 [dBm]",maximise,-84.1068,-83.8138,2.9301e-01,better,0.3484
6,"Cell-edge SINR, p05 [dB]",maximise,-5.8729,-5.8538,1.9010e-02,better,0.3237
7,"Median SINR, p50 [dB]",maximise,8.4932,8.3016,-1.9162e-01,worse,-2.2562
8,Served UE rate,maximise,0.5265,0.5758,4.9271e-02,better,9.3579
9,Peak PRB utilisation,minimise,0.7999,0.7999,-5.6731e-05,better,0.0071


Rule-based sweep


,KPI,Direction,Initial,Rule-based sweep,Change,Verdict,Improvement [%]
0,Coverage hole rate,minimise,0.1125,0.1044,-0.0081,better,7.1605
1,Co-band overlap rate,minimise,0.3164,0.3247,0.0083,worse,-2.6114
2,Overlap neighbours per covered tile,minimise,0.9625,1.1203,0.1578,worse,-16.3892
3,Weak coverage rate,minimise,0.3069,0.2538,-0.0531,better,17.2937
4,"Cell-edge RSRP, p05 [dBm]",maximise,-108.5644,-106.9844,1.5800,better,1.4554
5,"Median RSRP, p50 [dBm]",maximise,-84.1068,-80.8430,3.2638,better,3.8806
6,"Cell-edge SINR, p05 [dB]",maximise,-5.8729,-5.0047,0.8682,better,14.7832
7,"Median SINR, p50 [dB]",maximise,8.4932,9.8325,1.3393,better,15.7691
8,Served UE rate,maximise,0.5265,0.5930,0.0665,better,12.6342
9,Peak PRB utilisation,minimise,0.7999,0.7992,-0.0007,better,0.0884


,Method,Seed,Evaluations,Best evaluation,Ray tracing [min],Wall clock [min],Coverage hole rate,Co-band overlap rate,Overlap neighbours per covered tile,Weak coverage rate,"Cell-edge RSRP, p05 [dBm]","Median RSRP, p50 [dBm]","Cell-edge SINR, p05 [dB]","Median SINR, p50 [dB]",Served UE rate,Peak PRB utilisation,Cell load imbalance (CoV),Objective J,KPIs improved,KPIs worsened
0,Random search,42,145,62,7.4861,10.2194,0.1094,0.3368,1.0501,0.3063,-108.4613,-83.8138,-5.8538,8.3016,0.5758,0.7999,1.0805,0.8155,8,4
1,Rule-based sweep,42,28,11,1.0731,1.8539,0.1044,0.3247,1.1203,0.2538,-106.9844,-80.8430,-5.0047,9.8325,0.5930,0.7992,0.9416,0.8189,9,3


**Observations.** Both baselines raise J: +1.35 % for the sweep and +0.92 % for random search. The sweep improves
9 of the twelve reported measures and random search 8. **The sweep is the stronger configuration on almost every
view**: it cuts the hole rate 7.2 % against 2.7 %, weak coverage 17.3 % against 0.2 %, lifts median RSRP 3.3 dB
against 0.3 dB and median SINR 1.3 dB where random search *loses* 0.2 dB. It also serves more UEs (0.5930 against
0.5758).

**Neither baseline improves co-band overlap.** The sweep takes the overlap rate from 0.3164 to 0.3247 and random
search to 0.3368; both raise overlap neighbours per covered tile as well, the sweep by 16 % (0.963 to 1.120) and
random search by 9 % (to 1.050). Uptilting a whole band lifts every cell's footprint at once, so the footprints meet
in more places.

That is the objective's exchange rate, not an accident: closing a hole gains the full 1.000 of a tile's utility while
splitting a clean tile costs 0.264, so a search will accept nearly four newly crowded tiles per hole closed. Note
that overlap-reducing candidates *existed* - the sweep evaluated one reaching an overlap rate of 0.3067, better than
the incumbent - and J did not select it. On the one band J actually reads, 2600 MHz, overlap **improves** under both
methods (0.2010 to 0.1901 for the sweep and 0.1788 for random search); it is the two unscored layers and the newly
covered ground that push the band-collapsed rate up (notebook 04).

**Cell load imbalance worsens under both**, by 3.5 % and 18.8 %, as traffic concentrates on the cells that serve it
best. Nothing in the objective asks for even load, and ADR 0007 records that as a deliberate omission.

**Peak PRB utilisation does not move**: 0.7999 before, 0.7992-0.7999 after. Every configuration runs at the 0.8
admission ceiling, so the served-rate gains come from better SINR needing fewer PRBs per UE, not from spare capacity -
the median PRBs per served UE falls from 53.2 to about 43.4 under both.

## 11. Coverage Map Comparison

Best-server RSRP before and after, and the tiles that crossed the coverage-hole threshold; then the RSRP change of
each method's best configuration. Crimson triangles are the masts.

In [ ]:
best_rsrp = {"incumbent": max_rsrp(baseline_rsrp)}
best_rsrp.update({method: max_rsrp(best[method].radio_map["rsrp_dbm"].astype(float)) for method in METHODS})

for method in METHODS:
    figure = plots.coverage_maps(
        best_rsrp["incumbent"], best_rsrp[method], baseline, cfg, cells=cells, name=method
    )
    save_fig(figure, f"coverage_before_after_{method}")
    plt.show()

# No path in either map makes the change undefined; it stays blank.
with np.errstate(invalid="ignore"):
    change = {label(m): best_rsrp[m] - best_rsrp["incumbent"] for m in METHODS}
figure = plots.map_row(
    change, baseline, colorbar_label="Change in best-server RSRP [dB]", symmetric=True, cells=cells
)
save_fig(figure, "rsrp_change_maps")
plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2277759910.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2277759910.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2277759910.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Regions worth inspecting: the cell edges between the three corner nodes and the centre node, the tiles that opened or
closed a hole, and heavily overlapped areas near the masts.

**Observations.** Both baselines raise best-server RSRP almost everywhere, and the sweep does it most evenly: it
improves 95.7 % of reached tiles by a median of 3.2 dB, closing 820 hole tiles while opening 6. Random search is the
weaker and noisier of the two - 64.0 % of tiles improve, median +1.0 dB, 459 holes closed and 147 opened - because a
single Sobol draw uptilts some cells far more than their neighbours and trades one area against another. Note that
none of this signal gain scores: since ADR 0009 the objective reads only whether a cell clears -120 dBm and how many
others crowd it, so the sweep's 3.2 dB is visible in `rsrp_p50_dbm` and invisible in J.

## 12. KPI-Level Analysis

Coverage classes by area and by demand, overlap, and which band serves the UEs. Demand is the incumbent's, for every
configuration, so the weights do not move. The project has no band-priority score; the serving-band mix is read
against the preference order `kpi.capacity.band_preference` instead.

In [ ]:
configurations = {"incumbent": compare.configuration(baseline, ue, cfg)}
configurations.update({m: compare.configuration(best[m].radio_map, ue, cfg) for m in METHODS})
demand = configurations["incumbent"].demand

coverage = compare.coverage_comparison(
    {name: maps.coverage_table(c.rsrp, demand, cfg) for name, c in configurations.items()}
)
coverage.columns = ["coverage"] + [
    f"{label(key)}: {label(f'{share}_share')}"
    for key, share in (column.rsplit("_", 1) for column in coverage.columns[1:])
]
coverage = readable(coverage)
save_table(coverage, "coverage_by_area_and_demand")
display(coverage)

overlap = pd.DataFrame(
    {
        "configuration": list(configurations),
        "overlap_rate": [runs[0].incumbent_kpi.overlap_rate]
        + [best[m].best_kpi.overlap_rate for m in METHODS],
        "Mean overlapping neighbours": [mean_overlap_neighbors(c.rsrp) for c in configurations.values()],
    }
)
overlap = readable(overlap)
save_table(overlap, "overlap")
display(overlap)

band_labels = [str(band) for band in baseline["band_label"]]
print("Band preference, most preferred first:", ", ".join(label(b) for b in cfg.kpi.capacity.band_preference))
service = {name: compare.service_summary(c.served, band_labels) for name, c in configurations.items()}
service_table = readable(pd.DataFrame(service).T.rename_axis("configuration").reset_index())
save_table(service_table, "ue_service_summary")
display(service_table)
figure = plots.band_share_bars(service, band_labels)
save_fig(figure, "serving_band_mix")
plt.show()

tx_names = [str(name) for name in baseline["tx_name"]]
max_prb = CapacitySpec.from_config(cfg, band_labels, len(tx_names)).max_prb

# Every reported KPI, over all bands and per band, through the same functions:
# a band row is the definition given one band's layers, nothing new.
per_band = compare.band_kpis(configurations, band_labels, cfg)
save_table(readable(per_band), "band_kpis")
display(readable(per_band))
figure = plots.band_kpi_panels(
    per_band,
    ("hole_rate", "overlap_rate", "weak_rate", "rsrp_p05_dbm", "sinr_p05_db", "served_rate"),
)
save_fig(figure, "band_kpis")
plt.show()

# The series prb_utilisation_max and load_imbalance reduce. The admission
# ceiling bounds every cell in it, so a band running at the ceiling is a cell
# that refused traffic, not one that overloaded.
ceiling = float(cfg.kpi.capacity.max_admission_utilisation)
usage = compare.prb_usage_by_time(configurations, band_labels, tx_names, max_prb)
save_table(readable(usage), "prb_usage_by_time")
figure = plots.prb_usage_heatmaps(usage, ceiling)
save_fig(figure, "prb_usage")
plt.show()
print(f"peak utilisation over every cell-band and interval: {usage['utilisation'].max():.1%} "
      f"(admission ceiling {ceiling:.0%})")


,Coverage class,Current configuration: Share of area,Current configuration: Share of demand,Random search: Share of area,Random search: Share of demand,Rule-based sweep: Share of area,Rule-based sweep: Share of demand
0,hole,0.1125,0.000,0.1094,0.0037,0.1044,0.00
1,weak,0.3069,0.836,0.3063,0.8037,0.2538,0.77
2,good,0.5806,0.164,0.5843,0.1925,0.6417,0.23


,Configuration,Co-band overlap rate,Mean overlapping neighbours
0,Current configuration,0.3164,0.9625
1,Random search,0.3368,1.0501
2,Rule-based sweep,0.3247,1.1203


Band preference, most preferred first: 2600 MHz, 1800 MHz, 700 MHz


,Configuration,UE reports,Share not served,"Served SINR, 10th percentile [dB]","Served SINR, median [dB]","PRBs per served UE, median",Share served on 2600 MHz,Share served on 1800 MHz,Share served on 700 MHz
0,Current configuration,10087.0,0.4735,-0.3751,5.1257,53.1831,0.3408,0.1050,0.0807
1,Random search,10087.0,0.4242,0.1872,6.8475,43.6465,0.4113,0.0953,0.0692
2,Rule-based sweep,10087.0,0.4070,0.2463,6.9498,43.1678,0.4379,0.0816,0.0736


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2412617460.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Configuration,Band,Coverage hole rate,Co-band overlap rate,Overlap neighbours per covered tile,Weak coverage rate,"Cell-edge RSRP, p05 [dBm]","Median RSRP, p50 [dBm]","Cell-edge SINR, p05 [dB]","Median SINR, p50 [dB]",Served UE rate,Peak PRB utilisation,Cell load imbalance (CoV)
0,Current configuration,All bands,0.1125,0.3164,0.9625,0.3069,-108.5644,-84.1068,-5.8729,8.4932,0.5265,0.7999,0.9098
1,Current configuration,2600 MHz,0.2884,0.2010,0.3703,0.5124,-115.4094,-98.0890,-17.5911,-1.6299,0.3408,0.7999,0.6775
2,Current configuration,1800 MHz,0.2350,0.2067,0.3564,0.4171,-112.4538,-91.7208,-11.6703,4.1494,0.1050,0.7994,0.6074
3,Current configuration,700 MHz,0.1381,0.2396,0.3691,0.2945,-108.5333,-83.9182,-5.2264,8.3972,0.0807,0.7991,0.9986
4,Random search,All bands,0.1094,0.3368,1.0501,0.3063,-108.4613,-83.8138,-5.8538,8.3016,0.5758,0.7999,1.0805
5,Random search,2600 MHz,0.2631,0.1788,0.3366,0.4147,-113.9745,-92.3950,-16.1740,2.6907,0.4113,0.7995,0.7194
6,Random search,1800 MHz,0.2259,0.2064,0.3866,0.3644,-112.0356,-88.8772,-11.2691,5.5166,0.0953,0.7999,1.0369
7,Random search,700 MHz,0.1373,0.2496,0.4496,0.3002,-108.7323,-84.1073,-5.4767,7.7593,0.0692,0.7998,1.1554
8,Rule-based sweep,All bands,0.1044,0.3247,1.1203,0.2538,-106.9844,-80.8430,-5.0047,9.8325,0.5930,0.7992,0.9416
9,Rule-based sweep,2600 MHz,0.2618,0.1901,0.3385,0.4211,-114.0820,-92.7345,-16.2745,2.1617,0.4379,0.7992,0.5741


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2412617460.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


peak utilisation over every cell-band and interval: 80.0% (admission ceiling 80%)


C:\Users\nguye\AppData\Local\Temp\ipykernel_8020\2412617460.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Coverage holes

Both baselines cut the hole rate by area: 0.1125 to 0.1044 for the sweep and 0.1094 for random search. The current
configuration has no demand on hole tiles; the sweep still moves none onto them, and random search moves 0.37 %.

### Coverage overlap

Neither baseline improves overlap on the reported, band-collapsed measure. The sweep raises the overlap rate from
0.3164 to 0.3247 and the mean number of overlapping neighbours over covered tiles from 0.963 to 1.120; random search
raises the rate to 0.3368 and the mean to 1.050, but leaves fewer tiles with three or more neighbours (16.5 % against
the incumbent's 17.6 %).

Split by band the picture reverses on the layer that matters: 2600 MHz, the band the objective scores on 71 % of
tiles, falls from 0.2010 to 0.1901 under the sweep and 0.1788 under random search. The unscored 1800 and 700 MHz
layers stay flat or drift worse, and the newly covered ground arrives with neighbours of its own - together that is
what lifts the collapsed rate.

### Multi-band coordination

The share of UE reports served on 2600 MHz, the preferred band, grows from 34.1 % to 43.8 % for the sweep and 41.1 %
for random search, while the 1800 MHz share falls from 10.5 % to 8.2 % and 9.5 %, and 700 MHz from 8.1 % to 7.4 % and
6.9 %. The served ratio rises with it, and served SINR improves at the median (5.1 to 6.9 dB) and at the 10th
percentile (-0.4 to +0.2 dB). TuRBO does the opposite on the mid band - see notebook 03b.

### Weak coverage

Weak coverage falls by area for the sweep (30.7 % to 25.4 %) and barely moves for random search (30.6 %), and the
share of demand on weak tiles falls with it (83.6 % to 77.0 % and 80.4 %). It is still where most of the demand sits.

## 13. Key Observations

- The current per-band tilts (12°, 10°, 8°) are all downtilted relative to what either baseline prefers: both uptilt
  every band, and both gain - +3.7 % on J for the sweep and +3.0 % for random search.
- A structured three-variable sweep beat unstructured 36-variable random search on J at a seventh of the ray-tracing
  cost (0.9 against 5.9 minutes). Random search found its best at evaluation 7 of 145 and never improved after that:
  its budget is spread too thinly over a 36-dimensional box to add anything beyond the initial design.
- The gains are broad rather than narrow. The sweep improves the hole rate, weak coverage, both RSRP percentiles,
  both SINR percentiles and the served rate at once, and closes 761 hole tiles while opening 1.
- What neither fixes is overlap or load: both raise the overlap rate and the neighbour count, both leave peak
  utilisation at the admission ceiling, and both make the load less even. Overlap is the gap a per-cell-band search
  can close and a per-band sweep structurally cannot.
- The sweep drives 1800 MHz to 0°, the bottom of its tilt box, so its result is partly set by the bounds.
- These are one seed on one scenario; notebook 04 reports intervals over seeds.

## 14. Baseline Limitations

- **Random search** does not model the objective landscape or reuse what earlier evaluations revealed; with 36
  variables, a fixed budget covers the box only sparsely. Its best point here is evaluation 7 of 145.
- **The rule-based sweep** cannot give cells on one band different tilts, so it cannot fix a local problem without
  moving every cell on that band, and coordinate descent stops at the first configuration no single band can improve.
  It is also the method that raises the overlap neighbour count most, for the same reason.
- Every candidate costs a full ray trace, so either method's result depends on the budget it was given.
- The winner is the best of many evaluations under one solver seed, so its score is biased upward (notebook 04,
  threats to validity).
- Neither method constrains how far an antenna moves; tilt change is reported, not penalised, and both move all 36
  cell-bands.

## 15. Baselines vs. the Proposed Method

| Property | Random search | Rule-based sweep | TuRBO (notebook 03b) |
| --- | --- | --- | --- |
| Search strategy | Sobol sampling over the box | Shared tilt per band, coordinate descent | Trust-region Bayesian optimization |
| Surrogate model | No | No | Gaussian process |
| Uses previous evaluations | No | Only the current best | Yes, to fit the model and move the trust region |
| High-dimensional search | Uniform coverage, thins with dimension | Collapses 36 variables to 3 | Local trust region around the best point |
| Simulations | `n_init + n_iter` | `n_band · n_steps · n_rounds` at most | `n_init + n_iter`, as random search |
| Best objective | Section 10 | Section 10 | Notebook 04 |

All three share the initial network state, simulator, scenario, constraints and objective; random search and TuRBO
also share the evaluation budget and the initial Sobol design. The comparison with confidence intervals over seeds is
in notebook 04.

## 16. Conclusion

The baselines set a reproducible reference for multi-band tilt optimization on this scenario. They show:

- the improvement a simple strategy reaches, led by the rule-based sweep (+2.6 % on J in 28 evaluations);
- how tilt drives the objective here: the current configuration is over-downtilted, and uptilting every band buys
  coverage, signal quality and served traffic together;
- the two measures that resist both baselines - overlap neighbours per covered tile and cell load imbalance;
- that unstructured search over the 36-dimensional tilt space is inefficient at this budget.

TuRBO (notebook 03b) has to beat the rule-based sweep, not just random search, to justify its cost; notebook 04
makes that comparison.